**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Git & Collaboration

The tool every workshop in this repository was built with. Two sessions: how git *thinks* (snapshots, branches, merges — practiced in a sandbox repo this notebook creates), and the collaboration loop (fork → branch → PR → review) you'd use to contribute a workshop to this very curriculum.

## 1. Pre-requisites

A terminal with `git` installed (`git --version` to check). Runs on Linux/macOS/WSL — every cell uses `%%bash`.

In [1]:
%%bash
# a throwaway sandbox for the whole session — safe to delete and re-run anytime
rm -rf /tmp/sps_git_demo && mkdir -p /tmp/sps_git_demo && cd /tmp/sps_git_demo
git init -q -b main
git config user.name "SPS Student"
git config user.email "student@example.edu"
echo "sandbox ready: /tmp/sps_git_demo"

sandbox ready: /tmp/sps_git_demo


---
### 🕐 Session 1 of 2 — *Commits, Branches & Merges* (~40 min)
**Goal:** understand the snapshot graph; branch, merge, and resolve a real conflict.
**Feeds into:** Session 2 (the collaboration loop).

---

## 2. Git Thinks in Snapshots

💡 **Intuition.** A commit is a **snapshot of the entire project plus a pointer to its parent** — history is a linked list ([Data Structures in C](../../Intro_Programming/Data_Structures_in_C.ipynb)!) that branching turns into a DAG. A *branch* is nothing but a movable name-tag on one commit; creating one costs nothing. The staging area (`git add`) is the loading dock: you choose what enters the next snapshot, which is what makes small focused commits possible.

In [2]:
%%bash
cd /tmp/sps_git_demo
echo "fs = 1000" > config.py
echo "def filter(x): return x" > dsp.py
git add config.py dsp.py
git commit -q -m "initial: config + filter stub"

echo "def filter(x): return 2*x" > dsp.py
git add dsp.py && git commit -q -m "filter: apply gain"

git log --oneline

425f33b filter: apply gain
130f1f2 initial: config + filter stub


In [3]:
%%bash
cd /tmp/sps_git_demo
# branch = name-tag; switch = move HEAD; the working tree follows
git switch -q -c experiment
echo "def filter(x): return x**2   # nonlinear!" > dsp.py
git add dsp.py && git commit -q -m "experiment: nonlinear filter"

git switch -q main
echo "fs = 2000" > config.py
git add config.py && git commit -q -m "double the sample rate"

git log --oneline --graph --all

* 9c2f446 experiment: nonlinear filter
| * 32fb8d8 double the sample rate
|/  
* 425f33b filter: app

ly gain


* 130f1f2 initial: config + filter stub


### 2.1. Merging — and the Conflict You Shouldn't Fear

💡 **Intuition.** Merging combines two branches' changes. When they touched *different* files (or different lines), git does it silently. When both edited the same lines, git stops and asks — writing both versions into the file between `<<<<<<<` markers. A conflict is not an error; it's git refusing to guess. You edit, keep what's right, `add`, and `commit` — that's the entire skill.

In [4]:
%%bash
cd /tmp/sps_git_demo
git merge -q experiment -m "merge experiment (different files: automatic)" && echo "auto-merged cleanly"
cat dsp.py config.py

# now manufacture a real conflict: both branches edit the SAME line
git switch -q -c tune-a; echo "def filter(x): return 3*x**2" > dsp.py; git commit -qam "gain 3"
git switch -q main;      echo "def filter(x): return 5*x**2" > dsp.py; git commit -qam "gain 5"
git merge tune-a || true
echo "---- the conflicted file: ----"
cat dsp.py

auto-merged cleanly


def filter(x): return x**2   # nonlinear!
fs = 2000


Auto-merging dsp.py
CONFLICT (content): Merge conflict in dsp.py


Automatic merge failed; fix conflicts and then commit the result.
---- the conflicted file: ----


<<<<<<< HEAD
def filter(x): return 5*x**2
def filter(x): return 3*x**2
>>>>>>> tune-a


In [5]:
%%bash
cd /tmp/sps_git_demo
# resolve: keep one version (or write a third), then add + commit
echo "def filter(x): return 4*x**2   # compromise after discussion" > dsp.py
git add dsp.py && git commit -q -m "merge tune-a: settle gain at 4"
git log --oneline --graph | head -12

*   77676d1 merge tune-a: settle gain at 4
|\  
| * 56d4f52 gain 3
* | 0534f3e gain 5
|/  
*   7c30c

ca merge experiment (different files: automatic)
|\  
| * 9c2f446 experiment: nonlinear filter
* | 3

2fb8d8 double the sample rate
|/  
* 425f33b filter: apply gain
* 130f1f2 initial: config + filter s

tub


---
### 🕐 Session 2 of 2 — *The Collaboration Loop* (~35 min)
**Goal:** fork → branch → commit → PR → review: contribute to a shared repo without fear.
**Builds on:** Session 1.

---

## 3. Working With Others

The loop used by this curriculum (and most open source):

1. **Fork** the repo on GitHub (your own server-side copy) and `git clone` it.
2. **Branch** per contribution: `git switch -c add-wavelet-workshop`.
3. Commit in small, message-worthy steps. Good messages state *why*: `"split DSP S1: session ran 55 min in dry-run"` beats `"changes"`.
4. `git push origin add-wavelet-workshop`, then open a **pull request** — a proposed merge plus a discussion thread.
5. **Review**: comments, requested changes, more commits to the same branch (the PR updates automatically), then merge.

💡 **Intuition.** A PR is a *merge with witnesses*. The mechanics are Session 1's merge; everything else is social protocol that keeps `main` always-working: nobody — including the maintainer — edits `main` directly.

In [6]:
%%bash
cd /tmp/sps_git_demo
# simulate the receiving end of a PR: a feature branch merged into main with --no-ff
git switch -q -c add-notch-filter
echo "def notch(x): return x  # TODO" > notch.py
git add notch.py && git commit -q -m "add notch filter stub with tests plan"
git switch -q main
git merge -q --no-ff add-notch-filter -m "Merge PR #42: add notch filter (reviewed by 2)"
git log --oneline --graph | head -6

*   8d9cdb4 Merge PR #42: add notch filter (reviewed by 2)
|\  
| * 814c86c add notch filter stub wi

th tests plan
|/  
*   77676d1 merge tune-a: settle gain at 4
|\  


### 3.1. The Five Commands That Save You

| Situation | Command |
|---|---|
| "what changed?" | `git status` · `git diff` |
| "undo my *uncommitted* mess" | `git restore <file>` |
| "un-stage that" | `git restore --staged <file>` |
| "who wrote this line and why?" | `git blame <file>` → commit message |
| "I need last week's version" | `git log` → `git checkout <hash> -- <file>` |

And the golden safety fact: **anything committed is nearly impossible to lose** (`git reflog` remembers where every branch has ever pointed). Commit early, commit often.

In [7]:
%%bash
cd /tmp/sps_git_demo
git blame dsp.py | head -3
echo "..."
git reflog | head -5
rm -rf /tmp/sps_git_demo   # leave no trace

77676d1c (SPS Student 2026-07-22 13:26:21 -0400 1) def filter(x): return 4*x**2   # compromise after

 discussion


...


8d9cdb4 HEAD@{0}: merge add-notch-filter: Merge made by the 'ort' strategy.
77676d1 HEAD@{1}: checko

ut: moving from add-notch-filter to main
814c86c HEAD@{2}: commit: add notch filter stub with tests 

plan
77676d1 HEAD@{3}: checkout: moving from main to add-notch-filter
77676d1 HEAD@{4}: commit (merg

e): merge tune-a: settle gain at 4


## 4. Conclusion

Snapshots in a DAG, branches as name-tags, conflicts as questions not errors, and PRs as merges with witnesses. Your first real exercise: pick a 🚧 workshop from the [ROADMAP](../../ROADMAP.md), branch, and open a PR against this repository.

---
## Where next

- [Containers & Reproducibility](../Intro_Containers/Intro_Containers.ipynb) — versioning the *environment* the way git versions the code.
- [Intro to OS](../Intro_OS/Intro_OS.ipynb) — the filesystem tricks underneath `.git`.